In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [38]:
parsnp_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S3/parsnp_100_consensus/parsnp.vcf', delimiter='\t', skiprows=5)
ska_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S3/ska/alignment.vcf', delimiter='\t', skiprows=2)

parsnp_data = parsnp_data.drop(columns=['ancestral_sequence.fa.ref'])

## add .fasta to all cols in ska_data, remove . instances or replace as 0
ska_data.columns = (
    list(ska_data.columns[:9]) +
    [f"{col}.fasta" for col in ska_data.columns[9:]]
)

ska_data = ska_data[ska_data['ALT'] != '.']
ska_data = ska_data.replace('.', 0)

/tmp/ipykernel_3353702/1263833114.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ska_data = ska_data.replace('.', 0)


In [39]:
k = 21  # window size

pos = parsnp_data["POS"].to_numpy()
sample_cols = parsnp_data.columns[9:]

results = {}
close_locations = {}  # store tuples of nearby variant positions

num_variants_per_sample = []

for col in sample_cols:
    idx = np.where(parsnp_data[col].to_numpy() == 1)[0]
    num_variants_per_sample.append(len(idx))
    
    if len(idx) < 2:
        results[col] = 0
        close_locations[col] = []
        continue
    
    sample_pos = pos[idx]
    diffs = np.diff(sample_pos)
    
    threshold = k//2 + 1
    close_mask = diffs <= threshold
    
    # ---- collect tuples ----
    pairs = [
        (sample_pos[i], sample_pos[i+1])
        for i in np.where(close_mask)[0]
    ]
    close_locations[col] = pairs
    
    # ---- counting logic (your original method) ----
    count = (
        close_mask.sum() * 2
        - np.sum(close_mask[:-1] & close_mask[1:])
    )
    
    results[col] = count

result_series = pd.Series(results)

result_series
result_series.sum()

np.int64(12)

In [40]:
print(max(num_variants_per_sample))
print(min(num_variants_per_sample))
print(np.median(num_variants_per_sample))

28
6
18.0


In [41]:
k = 21  # window size

pos = parsnp_data["POS"].to_numpy()
sample_cols = parsnp_data.columns[10:]

results = {}
cluster_locations = {}

for col in sample_cols:
    idx = np.where(parsnp_data[col].to_numpy() == 1)[0]
    
    if len(idx) < 3:
        results[col] = 0
        cluster_locations[col] = []
        continue
    
    sample_pos = pos[idx]
    
    clusters = []
    counted = set()
    
    left = 0
    for right in range(len(sample_pos)):
        
        # shrink window if too large
        while sample_pos[right] - sample_pos[left] > k:
            left += 1
        
        window_size = right - left + 1
        
        if window_size >= 3:
            cluster = tuple(sample_pos[left:right+1])
            clusters.append(cluster)
            counted.update(cluster)
    
    cluster_locations[col] = clusters
    results[col] = len(counted)

result_series = pd.Series(results)

result_series
result_series.sum()

np.int64(0)

In [42]:
ska_data

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,0-2-0.189533.fasta,...,58-69-0.368930.fasta,60-81-0.279477.fasta,62-70-0.407166.fasta,64-40-0.015648.fasta,6-45-0.495324.fasta,66-44-0.066319.fasta,68-60-0.452212.fasta,70-84-0.030962.fasta,72-77-0.425471.fasta,8-79-0.464173.fasta
12,Ancestral,106,0,C,T,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,1
13,Ancestral,111,0,G,A,0,0,0,GT,0,...,0,0,0,0,0,0,0,1,1,0
14,Ancestral,112,0,C,T,0,0,0,GT,0,...,0,0,0,0,1,0,0,0,0,0
15,Ancestral,138,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,1,0,0
16,Ancestral,165,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,Ancestral,8924,0,A,C,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
314,Ancestral,8955,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
324,Ancestral,8973,0,A,G,0,0,0,GT,0,...,1,1,1,1,0,1,1,0,0,0
332,Ancestral,9068,0,G,A,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0


In [43]:
parsnp_data

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,18-18-0.156890.fasta,...,62-70-0.407166.fasta,46-54-0.035823.fasta,14-48-0.410520.fasta,50-64-0.389780.fasta,22-91-0.348475.fasta,34-92-0.190368.fasta,36-74-0.276772.fasta,4-63-0.277289.fasta,10-15-0.185358.fasta,20-31-0.225142.fasta
0,Ancestral,106,AGAACATCGG.CCGATGCGGT,C,T,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,1,0,0,0
1,Ancestral,111,ATCGGCCGAT.GCGGTATGGT,G,A,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
2,Ancestral,112,TCGGCCGATG.CGGTATGGTA,C,T,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
3,Ancestral,138,TTATTTTTGT.ATTTCAAATA,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
4,Ancestral,165,TCGGATATTC.AACGTCATAA,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,Ancestral,8973,GTAGTTAGGA.AAAGCAACGG,A,G,40,PASS,NaN,GT,0,...,1,1,0,1,0,0,0,0,0,0
247,Ancestral,8978,TAGGAAAAGC.AACGGGTATG,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,1,0,0,0
248,Ancestral,8980,GGAAAAGCAA.CGGGTATGAA,C,T,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
249,Ancestral,9068,TATGGAAAAA.GGGGGGAGTA,G,A,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
len([col for col in parsnp_data.columns if col.endswith(".fasta")])

37

In [45]:
len([col for col in ska_data.columns if col.endswith(".fasta")])

37

In [46]:
def split_multiple_alt_alleles(df):

    df = df.copy()
    fasta_cols = df.columns[9:]  # sample columns

    multi_mask = df["ALT"].str.contains(",", na=False)

    # Separate multi-ALT rows
    df_multi = df[multi_mask].copy()
    df_single = df[~multi_mask].copy()

    new_rows = []

    for _, row in df_multi.iterrows():

        alts = row["ALT"].split(",")

        for i, alt in enumerate(alts, start=1):  # ALT index = 1-based

            new_row = row.copy()
            new_row["ALT"] = alt

            # Convert sample columns:
            # keep only values equal to this alt index
            for col in fasta_cols:
                new_row[col] = 1 if row[col] == i else 0

            new_rows.append(new_row)

    df_multi_split = pd.DataFrame(new_rows)

    # Combine back
    result = pd.concat([df_single, df_multi_split], ignore_index=True)

    return result

In [47]:
parsnp_data['POS'] = parsnp_data['POS'].astype(int)
ska_data['POS'] = ska_data['POS'].astype(int)

parsnp_data = split_multiple_alt_alleles(parsnp_data)
ska_data = split_multiple_alt_alleles(ska_data)

In [48]:
import os
import pandas as pd


def get_bronko_df(location):
    vcf_files = [f for f in os.listdir(location) if f.endswith(".vcf")]

    variant_dict = {}

    for vcf_file in vcf_files:
        sample_name = vcf_file.replace(".vcf", ".fasta")
        with open(os.path.join(location, vcf_file)) as f:
            for line in f:
                if line.startswith("#"):
                    continue
                fields = line.strip().split("\t")
                chrom, pos, _, ref, alt = fields[:5]
                key = (chrom, pos, ref, alt)
                if key not in variant_dict:
                    variant_dict[key] = {}
                variant_dict[key][sample_name] = 1
                
    print(variant_dict)

    # convert to DataFrame

    bronko = pd.DataFrame.from_dict(variant_dict, orient="index")
    bronko = bronko.fillna(0).astype(int)
    bronko.reset_index(inplace=True)
    bronko.rename(columns={"index": ["CHROM", "POS", "REF", "ALT"]}, inplace=True)

    new_names = ['#CHROM','POS','REF','ALT']
    bronko.columns = new_names + bronko.columns[4:].tolist()
    bronko['POS'] = bronko['POS'].astype(int)
    return bronko

bronko = get_bronko_df("/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S3/bronko_single_ref")  # change this to your VCF folder path
bronko

# df = bronko.drop(columns='index')

# # save table

# # df.to_csv("variant_presence_matrix.csv", index=False)
# print("Saved variant presence matrix to variant_presence_matrix.csv")


{('Ancestral', '389', 'C', 'T'): {'16-6-0.396032.fasta': 1}, ('Ancestral', '1169', 'T', 'C'): {'16-6-0.396032.fasta': 1}, ('Ancestral', '1566', 'G', 'A'): {'16-6-0.396032.fasta': 1, '18-18-0.156890.fasta': 1, '34-92-0.190368.fasta': 1, '30-46-0.092426.fasta': 1, '28-52-0.191228.fasta': 1, '26-11-0.161928.fasta': 1, '22-91-0.348475.fasta': 1, '24-68-0.272818.fasta': 1, '36-74-0.276772.fasta': 1, '32-62-0.077237.fasta': 1, '20-31-0.225142.fasta': 1}, ('Ancestral', '1957', 'C', 'T'): {'16-6-0.396032.fasta': 1, '18-18-0.156890.fasta': 1, '34-92-0.190368.fasta': 1, '30-46-0.092426.fasta': 1, '28-52-0.191228.fasta': 1, '26-11-0.161928.fasta': 1, '22-91-0.348475.fasta': 1, '24-68-0.272818.fasta': 1, '36-74-0.276772.fasta': 1, '32-62-0.077237.fasta': 1, '20-31-0.225142.fasta': 1}, ('Ancestral', '2654', 'A', 'G'): {'16-6-0.396032.fasta': 1, '18-18-0.156890.fasta': 1, '34-92-0.190368.fasta': 1, '30-46-0.092426.fasta': 1, '28-52-0.191228.fasta': 1, '26-11-0.161928.fasta': 1, '22-91-0.348475.fasta

,#CHROM,POS,REF,ALT,16-6-0.396032.fasta,18-18-0.156890.fasta,34-92-0.190368.fasta,30-46-0.092426.fasta,28-52-0.191228.fasta,26-11-0.161928.fasta,...,4-63-0.277289.fasta,0-2-0.189533.fasta,10-15-0.185358.fasta,2-53-0.380368.fasta,14-48-0.410520.fasta,8-79-0.464173.fasta,12-34-0.325192.fasta,6-45-0.495324.fasta,70-84-0.030962.fasta,72-77-0.425471.fasta
0,Ancestral,389,C,T,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Ancestral,1169,T,C,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Ancestral,1566,G,A,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,Ancestral,1957,C,T,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,Ancestral,2654,A,G,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,Ancestral,6320,A,G,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
250,Ancestral,6977,G,A,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
251,Ancestral,7430,G,A,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
252,Ancestral,7546,C,T,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1


In [49]:
def get_shared(parsnp_data, ska_data, bronko):
    df1 = parsnp_data
    df2 = ska_data
    df3 = bronko

    variant_cols = ["#CHROM", "POS", "REF", "ALT"]

    def add_suffix(df, suffix):
        sample_cols = [c for c in df.columns if c not in variant_cols]
        return df.rename(columns={c: f"{c}_{suffix}" for c in sample_cols})

    df1_ = add_suffix(df1, "1")
    df2_ = add_suffix(df2, "2")
    df3_ = add_suffix(df3, "3")

    merged = (
        df1_
        .merge(df2_, on=variant_cols, how="outer")
        .merge(df3_, on=variant_cols, how="outer")
    )
    

    fasta_cols = [c for c in merged.columns if ".fasta" in c]

    merged[fasta_cols] = (
        merged[fasta_cols]
        .fillna(0)
        .astype(int)
    )

    sample_cols = [c for c in df1.columns[9:] if c not in variant_cols]


    
    results = []
    ska_missed = []

    for c in sample_cols:
        col1 = f"{c}_1"
        col2 = f"{c}_2"
        col3 = f"{c}_3"

        a = merged[col1] == 1
        b = merged[col2] == 1
        d = merged[col3] == 1

        only_1 = (a & ~b & ~d).sum()
        only_2 = (~a & b & ~d).sum()
        only_3 = (~a & ~b & d).sum()

        shared_12_only = (a & b & ~d).sum()
        shared_13_only = (a & ~b & d).sum()
        shared_23_only = (~a & b & d).sum()

        shared_123 = (a & b & d).sum()
        
        if shared_13_only:
            print(c)
            ska_missed.append(c)

        results.append({
            "sample": c,
            "only_1": int(only_1),
            "only_2": int(only_2),
            "only_3": int(only_3),
            "shared_12_only": int(shared_12_only),
            "shared_13_only": int(shared_13_only),
            "shared_23_only": int(shared_23_only),
            "shared_123": int(shared_123)
        })

    overlap_df = pd.DataFrame(results)
    overlap_df
    return overlap_df, merged, ska_missed

In [50]:
overlap_df, merged, ska_missed = get_shared(parsnp_data, ska_data, bronko)
overlap_df

24-68-0.272818.fasta
48-93-0.236361.fasta
16-6-0.396032.fasta
26-11-0.161928.fasta
6-45-0.495324.fasta
38-7-0.430798.fasta
8-79-0.464173.fasta
2-53-0.380368.fasta
54-27-0.410672.fasta
0-2-0.189533.fasta
12-34-0.325192.fasta
14-48-0.410520.fasta
36-74-0.276772.fasta
4-63-0.277289.fasta
10-15-0.185358.fasta


,sample,only_1,only_2,only_3,shared_12_only,shared_13_only,shared_23_only,shared_123
0,18-18-0.156890.fasta,0,0,0,0,0,0,20
1,44-38-0.325374.fasta,0,0,0,0,0,0,18
2,30-46-0.092426.fasta,0,0,0,0,0,0,18
3,24-68-0.272818.fasta,0,0,0,0,3,0,17
4,48-93-0.236361.fasta,0,0,0,0,2,0,10
5,56-42-0.323257.fasta,0,0,0,0,0,0,9
6,58-69-0.368930.fasta,0,0,0,0,0,0,18
7,16-6-0.396032.fasta,0,0,0,0,1,0,19
8,52-98-0.006603.fasta,0,0,0,0,0,0,12
9,60-81-0.279477.fasta,0,0,0,0,0,0,10


In [51]:
overlap_df.iloc[:, 1:].sum()

only_1              0
only_2              0
only_3              0
shared_12_only      0
shared_13_only     37
shared_23_only      0
shared_123        627
dtype: int64

In [163]:
results = {}

for sample in ska_missed:
    cols = ["POS", "REF", "ALT"] + [f"{sample}_{i}" for i in [1,2,3]]
    results[sample] = merged[merged[f"{sample}_1"] == 1][cols]

final_df = (
    pd.concat(results, names=["sample"])
      .reset_index(level=0)
      .reset_index(drop=True)
)
final_df

,sample,POS,REF,ALT,24-68-0.272818.fasta_1,24-68-0.272818.fasta_2,24-68-0.272818.fasta_3,48-93-0.236361.fasta_1,48-93-0.236361.fasta_2,48-93-0.236361.fasta_3,...,14-48-0.410520.fasta_3,36-74-0.276772.fasta_1,36-74-0.276772.fasta_2,36-74-0.276772.fasta_3,4-63-0.277289.fasta_1,4-63-0.277289.fasta_2,4-63-0.277289.fasta_3,10-15-0.185358.fasta_1,10-15-0.185358.fasta_2,10-15-0.185358.fasta_3
0,24-68-0.272818.fasta,1566,G,A,1.0,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,24-68-0.272818.fasta,1957,C,T,1.0,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,24-68-0.272818.fasta,2590,G,A,1.0,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,24-68-0.272818.fasta,2654,A,G,1.0,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,24-68-0.272818.fasta,3382,G,C,1.0,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,10-15-0.185358.fasta,7739,A,G,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0
329,10-15-0.185358.fasta,8105,G,A,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0
330,10-15-0.185358.fasta,8227,T,C,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0
331,10-15-0.185358.fasta,8253,T,C,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0


In [164]:
ska_data[ska_data['POS']==1566][['POS', 'REF', 'ALT', '24-68-0.272818.fasta']]

,POS,REF,ALT,24-68-0.272818.fasta
46,1566,G,A,1
